**Step 4: Compute the Name Signals**

This notebook computes, once, everything the candidate search methods know about a name, so that the experiments (steps 5–8) only compare and rank. For every name spelling in the archive, the dictionaries and the queries, it computes:

1. its **embedding** under each of the four embedding models: E1, the byte-level name encoder of Jumle et al.; E2, Symphonym v7; E3a, LaBSE; E3b, multilingual-e5-small. An embedding is a vector; spellings of the same name in either script should lie close together
2. its **phonetic codes**: Daitch–Mokotoff codes (computed by PostgreSQL, Hebrew romanised with AnyAscii first, L3) and Beider–Morse tokens (Abydos, L4)
3. the **trigram similarity** of every query token to every archive spelling of the same script (PostgreSQL `pg_trgm`, L2)
4. the **time each method takes to encode each query**, used for the latency of step 8

The models are downloaded on first use: E1's repository into the git-ignored `./assets`, the others into `~/.cache/huggingface`. PostgreSQL runs as a private cluster in the git-ignored `./cache/pgdata`. It reads `./outputs/step_1_persons.csv`, `./outputs/step_2_queries.csv` and `./outputs/step_3_dictionary_d3.csv`, and outputs:

1. `./outputs/step_4_embeddings_<model>.npz`: the texts and their unit-length vectors, for `e1`, `e2`, `e3a` and `e3b`
2. `./outputs/step_4_name_codes.csv`: every spelling with its display spelling, whether it is in the archive, and its phonetic codes
3. `./outputs/step_4_trigram_similarities.npz`: the non-zero trigram similarities between query tokens and archive spellings
4. `./outputs/step_4_encoding_times.csv`: each method's time to encode each query, in milliseconds

**1. Configuration and Vocabulary**

This cell defines the settings and builds the vocabulary every method runs over:

1. `archive_forms`: each distinct normalised spelling of the persons' tokens, with one display spelling, its most frequent recorded one
2. `texts_to_embed`: every display spelling of the archive, the dictionaries and the queries
3. `spellings_to_code`: every (script, normalised form) the phonetic methods must code

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration and vocabulary  —  what every method must encode
#
# Embeddings are keyed by display text and phonetic codes by normalised
# form. A form recorded with several spellings (José, Jose) is embedded
# through its most frequent one, so each archive form has one vector.
# ─────────────────────────────────────────────────────────────────────────────
import csv
import importlib.util
import json
import subprocess
import sys
import tempfile
import time
from collections import Counter, defaultdict
from pathlib import Path

import abydos.phonetic as bm_phonetic
import numpy as np
import psycopg
import torch
from abydos.phonetic._beider_morse_data import BMDATA
from anyascii import anyascii
from huggingface_hub import snapshot_download
from safetensors.numpy import load_file
from sentence_transformers import SentenceTransformer

# ═════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═════════════════════════════════════════════════════════════════════════════
OUTPUTS_DIR   = Path("outputs")                    # every step's outputs
E1_REPO       = "https://github.com/vedant-jumle/cross-language-phonetic-text-alignment"
E1_CHECKPOINT = Path("assets/e1_repo/checkpoints/best_v2/model.safetensors")
E2_MODEL      = "docuracy/symphonym-v7"            # Hugging Face model ids
E3_MODELS     = {"e3a": ("sentence-transformers/LaBSE", ""),
                 "e3b": ("intfloat/multilingual-e5-small", "query: ")}   # (model, prefix)
PG_BIN        = Path(sys.executable).parent        # pg_ctl and initdb of the conda env
PG_DATA_DIR   = Path("cache/pgdata")               # the private cluster (git-ignored)
PG_PORT       = 5433                               # names the socket file only; no TCP
LATIN_LANGUAGES = "spanish,german,polish,russian,english"   # Beider–Morse, Latin input
RULE          = "─" * 72


def read_csv(path: Path) -> list[dict]:
    """Read the CSV file `path` into a list of dictionaries."""
    with path.open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


def start_postgres() -> Path:
    """Initialise (once) and start the private PostgreSQL cluster on a Unix socket.

    The en_US.UTF-8 locale makes pg_trgm treat Hebrew letters as letters.

    Returns:
        The socket folder, to pass to psycopg as `host`.
    """
    if not (PG_DATA_DIR / "PG_VERSION").exists():
        PG_DATA_DIR.mkdir(parents=True, exist_ok=True)
        subprocess.run([PG_BIN / "initdb", "-D", PG_DATA_DIR, "-U", "postgres", "--auth=trust",
                        "--encoding=UTF8", "--locale=en_US.UTF-8"], check=True, capture_output=True)
    stop_postgres()
    socket_dir = Path(tempfile.mkdtemp(prefix="pg"))
    subprocess.run([PG_BIN / "pg_ctl", "-D", PG_DATA_DIR, "-l", PG_DATA_DIR / "server.log", "-w",
                    "-o", f"-k {socket_dir} -c listen_addresses= -p {PG_PORT}", "start"],
                   check=True, capture_output=True)
    return socket_dir


def stop_postgres() -> None:
    """Stop the private cluster if it is running."""
    subprocess.run([PG_BIN / "pg_ctl", "-D", PG_DATA_DIR, "-m", "fast", "stop"],
                   check=False, capture_output=True)


persons = read_csv(OUTPUTS_DIR / "step_1_persons.csv")
queries = read_csv(OUTPUTS_DIR / "step_2_queries.csv")
dictionary = read_csv(OUTPUTS_DIR / "step_3_dictionary_d3.csv")

_spellings = defaultdict(Counter)
for _person in persons:
    for _token in json.loads(_person["tokens"]):
        _spellings[(_token["script"], _token["norm"])][_token["display"]] += 1
archive_forms = {key: sorted(counts.items(), key=lambda item: (-item[1], item[0]))[0][0]
                 for key, counts in sorted(_spellings.items())}
query_tokens = [token for q in queries for token in json.loads(q["tokens"])]
texts_to_embed = sorted(set(archive_forms.values()) | {r["display"] for r in dictionary}
                        | {t["display"] for t in query_tokens})
spellings_to_code = sorted(set(archive_forms) | {(r["script"], r["norm"]) for r in dictionary}
                           | {(t["script"], t["norm"]) for t in query_tokens})
print(f"✓ Configuration and vocabulary loaded ({len(archive_forms)} archive spellings, "
      f"{len(texts_to_embed)} texts to embed, {len(spellings_to_code)} spellings to code).")

/Users/andreiflorian/Development/cos333/cos333_gauchos_judios/search_algorithm/.conda/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Configuration and vocabulary loaded (1206 archive spellings, 2043 texts to embed, 2038 spellings to code).


**2. Embeddings**

This cell embeds every text of `texts_to_embed` with each of the four models, as unit-length vectors, so that the dot product of two vectors is their cosine similarity:

1. **E1** runs through a NumPy translation of its network, on the text's UTF-8 bytes, which is how a web service would run it without PyTorch
2. **E2** loads through Symphonym's own `inference.py`, whose attention-pooling layer is replaced by the definition the authors ship with v8, the one its v7 weights were trained with (the shipped v7 layer cannot load them). It is told the language: `he` for Hebrew script, `es` for Latin
3. **E3a** and **E3b** load through sentence-transformers; E3b's inputs start with `query: `, as its model card requires

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Embeddings  —  E1, E2, E3a and E3b, one vector per display spelling
# ─────────────────────────────────────────────────────────────────────────────
if "texts_to_embed" not in globals():
    raise RuntimeError("Run the 'Configuration and Vocabulary' cell first.")


def layer_norm(x: np.ndarray, gain: np.ndarray, bias: np.ndarray) -> np.ndarray:
    """Normalise the last axis of `x`, then scale by `gain` and shift by `bias`."""
    mean = x.mean(-1, keepdims=True)
    return (x - mean) / np.sqrt(((x - mean) ** 2).mean(-1, keepdims=True) + 1e-5) * gain + bias


def e1_layer(w: dict, x: np.ndarray, layer: int) -> np.ndarray:
    """Apply one pre-norm transformer layer of E1 (8 attention heads, ReLU) to `x`."""
    p, heads = f"transformer.layers.{layer}.", 8
    h = layer_norm(x, w[p + "norm1.weight"], w[p + "norm1.bias"])
    qkv = h @ w[p + "self_attn.in_proj_weight"].T + w[p + "self_attn.in_proj_bias"]
    q, k, v = (t.reshape(len(x), heads, -1).transpose(1, 0, 2) for t in np.split(qkv, 3, -1))
    scores = np.exp(q @ k.transpose(0, 2, 1) / np.sqrt(q.shape[-1]))
    attended = ((scores / scores.sum(-1, keepdims=True)) @ v).transpose(1, 0, 2).reshape(x.shape)
    x = x + attended @ w[p + "self_attn.out_proj.weight"].T + w[p + "self_attn.out_proj.bias"]
    h = layer_norm(x, w[p + "norm2.weight"], w[p + "norm2.bias"])
    hidden = np.maximum(0, h @ w[p + "linear1.weight"].T + w[p + "linear1.bias"])
    return x + hidden @ w[p + "linear2.weight"].T + w[p + "linear2.bias"]


def e1_encode(w: dict, text: str) -> np.ndarray:
    """Return E1's embedding of `text`: six layers over its UTF-8 bytes, averaged."""
    ids = list(text.encode("utf-8"))[:256]
    x = w["embedding.weight"][ids] + w["pos_embedding.weight"][:len(ids)]
    for layer in range(6):
        x = e1_layer(w, x, layer)
    pooled = x.mean(0)
    return pooled / np.linalg.norm(pooled)


def load_symphonym(model_dir: Path):
    """Load Symphonym v7 with the attention pooling its weights were trained with."""

    class AttentionPooling(torch.nn.Module):
        """Attention pooling as trained: Linear → Tanh → Dropout → Linear."""

        def __init__(self, hidden_dim: int, dropout: float = 0.1):
            """Build the scorer for states of size `hidden_dim`."""
            super().__init__()
            self.attention = torch.nn.Sequential(
                torch.nn.Linear(hidden_dim, hidden_dim // 2), torch.nn.Tanh(),
                torch.nn.Dropout(dropout), torch.nn.Linear(hidden_dim // 2, 1))

        def forward(self, x, mask=None):
            """Return the attention-weighted mean of `x` and the weights."""
            scores = self.attention(x).squeeze(-1)
            if mask is not None:
                scores = scores.masked_fill(~mask, float("-inf"))
            weights = torch.softmax(scores, dim=-1)
            return torch.bmm(weights.unsqueeze(1), x).squeeze(1), weights

    spec = importlib.util.spec_from_file_location("symphonym", model_dir / "inference.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.AttentionPooling = AttentionPooling
    return module.SymphonymModel(model_dir=model_dir)


def load_encoders() -> dict:
    """Load the four models; return, per model, a function embedding a list of texts."""
    if not E1_CHECKPOINT.exists():
        subprocess.run(["git", "clone", "--depth", "1", E1_REPO, "assets/e1_repo"], check=True)
    e1_weights = load_file(str(E1_CHECKPOINT))
    symphonym = load_symphonym(Path(snapshot_download(E2_MODEL, allow_patterns=[
        "config.json", "inference.py", "model.safetensors", "vocab/*"])))
    models = {"e1": lambda texts: np.stack([e1_encode(e1_weights, t) for t in texts]),
              "e2": lambda texts: symphonym.batch_embed(
                  [(t, "he" if any(0x590 <= ord(c) <= 0x5FF for c in t) else "es")
                   for t in texts])}
    for name, (model_id, prefix) in E3_MODELS.items():
        model = SentenceTransformer(model_id, device="cpu")
        models[name] = (lambda texts, m=model, p=prefix:
                        m.encode([p + t for t in texts], normalize_embeddings=True))
    return models


encoders = load_encoders()
print(f"{RULE}\nEmbeddings: {len(texts_to_embed)} texts\n{RULE}")
for _name, _encode in encoders.items():
    _vectors = np.concatenate([np.asarray(_encode(texts_to_embed[i:i + 256]), dtype=np.float32)
                               for i in range(0, len(texts_to_embed), 256)])
    _vectors /= np.linalg.norm(_vectors, axis=1, keepdims=True)
    np.savez_compressed(OUTPUTS_DIR / f"step_4_embeddings_{_name}.npz",
                        texts=np.array(texts_to_embed), vectors=_vectors)
    _pair = np.asarray(_encode(["Rebeca", "רבקה"]), dtype=np.float32)
    _pair /= np.linalg.norm(_pair, axis=1, keepdims=True)
    print(f"  {_name:<4} {_vectors.shape[1]:>4} dimensions   cosine(Rebeca, רבקה) "
          f"{float(_pair[0] @ _pair[1]):.3f}")
print(f"{RULE}\n  {len(encoders)} embedding files written")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 1312.09it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4016.55it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

────────────────────────────────────────────────────────────────────────
Embeddings: 2043 texts
────────────────────────────────────────────────────────────────────────


  e1    256 dimensions   cosine(Rebeca, רבקה) 0.889


  e2    128 dimensions   cosine(Rebeca, רבקה) 0.776


  e3a   768 dimensions   cosine(Rebeca, רבקה) 0.812


  e3b   384 dimensions   cosine(Rebeca, רבקה) 0.817
────────────────────────────────────────────────────────────────────────
  4 embedding files written


**3. Phonetic Codes and Trigram Similarities**

This cell computes the lexical signals. PostgreSQL computes the Daitch–Mokotoff codes of every spelling (`fuzzystrmatch`), Hebrew being romanised with AnyAscii first, and the trigram similarity (`pg_trgm`) of every query token to every archive spelling of the same script. Abydos computes the Beider–Morse tokens with the Ashkenazic rules, in its approximate pass and its Hebrew-value pass, which the library ships but does not expose, so the private match mode is set directly. It writes `./outputs/step_4_name_codes.csv` and `./outputs/step_4_trigram_similarities.npz`.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Phonetic codes and trigram similarities  —  PostgreSQL and Abydos
# ─────────────────────────────────────────────────────────────────────────────
if "encoders" not in globals():
    raise RuntimeError("Run the 'Embeddings' cell first.")


def beider_morse_encoders() -> dict:
    """Return the Beider–Morse encoders, keyed `<script>_<pass>`, with the
    Hebrew-value pass enabled for both scripts."""
    for language in BMDATA["ash"]["approx"]:
        if language != "common" and language not in BMDATA["ash"]["hebrew"]:
            BMDATA["ash"]["hebrew"][language] = ()
    coders = {}
    for script, language in (("latin", LATIN_LANGUAGES), ("hebrew", 0)):
        for match_mode in ("approx", "hebrew"):
            coder = bm_phonetic.BeiderMorse(name_mode="ash", language_arg=language)
            coder._match_mode = match_mode  # pylint: disable=protected-access
            coders[f"{script}_{match_mode}"] = coder
    return coders


def dm_input(script: str, norm: str) -> str:
    """Return the text PostgreSQL codes: the form itself, or its AnyAscii romanisation."""
    return anyascii(norm) if script == "hebrew" else norm


beider_morse = beider_morse_encoders()
_query_keys = sorted({(t["script"], t["norm"]) for t in query_tokens})
_archive_keys = sorted(archive_forms)
_socket = start_postgres()
with psycopg.connect(host=str(_socket), port=PG_PORT, user="postgres", dbname="postgres",
                     autocommit=True) as _connection:
    _connection.execute("CREATE EXTENSION IF NOT EXISTS fuzzystrmatch")
    _connection.execute("CREATE EXTENSION IF NOT EXISTS pg_trgm")
    _dm = _connection.execute("SELECT daitch_mokotoff(x) FROM unnest(%s::text[]) AS x",
                              [[dm_input(*k) for k in spellings_to_code]]).fetchall()
    _trigrams = _connection.execute(
        "SELECT q.i, a.j, similarity(q.norm, a.norm) FROM unnest(%s::text[], %s::text[]) "
        "WITH ORDINALITY AS q(script, norm, i) JOIN unnest(%s::text[], %s::text[]) "
        "WITH ORDINALITY AS a(script, norm, j) ON q.script = a.script "
        "WHERE similarity(q.norm, a.norm) > 0",
        [[s for s, _ in _query_keys], [n for _, n in _query_keys],
         [s for s, _ in _archive_keys], [n for _, n in _archive_keys]]).fetchall()
stop_postgres()

name_codes = [{"script": script, "norm": norm, "display": archive_forms.get((script, norm), ""),
               "in_archive": int((script, norm) in archive_forms),
               "dm_codes": " ".join(sorted(set(codes or []) - {"000000"})),
               **{f"bmpm_{mode}": " ".join(sorted(set(
                   beider_morse[f"{script}_{mode}"].encode(norm).split())))
                  for mode in ("approx", "hebrew")}}
              for (script, norm), (codes,) in zip(spellings_to_code, _dm)]
with (OUTPUTS_DIR / "step_4_name_codes.csv").open("w", encoding="utf-8", newline="") as _file:
    _writer = csv.DictWriter(_file, fieldnames=list(name_codes[0]), lineterminator="\n")
    _writer.writeheader()
    _writer.writerows(name_codes)
_triples = np.array(_trigrams, dtype=np.float64).reshape(-1, 3)
np.savez_compressed(OUTPUTS_DIR / "step_4_trigram_similarities.npz",
                    query_keys=np.array([f"{s}|{n}" for s, n in _query_keys]),
                    archive_keys=np.array([f"{s}|{n}" for s, n in _archive_keys]),
                    rows=_triples[:, 0].astype(int) - 1, cols=_triples[:, 1].astype(int) - 1,
                    values=_triples[:, 2].astype(np.float32))

print(f"{RULE}\nPhonetic codes and trigram similarities\n{RULE}")
print(f"  Daitch–Mokotoff   {sum(bool(r['dm_codes']) for r in name_codes):>6} spellings coded")
print(f"  Beider–Morse      {sum(bool(r['bmpm_approx']) for r in name_codes):>6} spellings coded")
print(f"  trigrams          {len(_triples):>6} query–archive pairs above zero")
print(f"{RULE}\n  written to step_4_name_codes.csv and step_4_trigram_similarities.npz")

────────────────────────────────────────────────────────────────────────
Phonetic codes and trigram similarities
────────────────────────────────────────────────────────────────────────
  Daitch–Mokotoff     2036 spellings coded
  Beider–Morse        2038 spellings coded
  trigrams           53040 query–archive pairs above zero
────────────────────────────────────────────────────────────────────────
  written to step_4_name_codes.csv and step_4_trigram_similarities.npz


**4. Encoding Times**

This cell times, for every query, the work each method does to encode it before any matching, which step 8 adds to the matching time to report the search's latency. The embedding models embed the query's tokens; L2 and L3 run the look-up a web service would run against a table of the archive's spellings (a trigram scan of the spellings of the token's script, and an indexed Daitch–Mokotoff look-up); L4 computes the Beider–Morse tokens; L1 needs only the normalisation. Each method answers 20 untimed queries first, so the times are those of a warm service. It writes `./outputs/step_4_encoding_times.csv`.

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Encoding times  —  warm, one query at a time, in milliseconds
# ─────────────────────────────────────────────────────────────────────────────
if "name_codes" not in globals():
    raise RuntimeError("Run the 'Phonetic Codes and Trigram Similarities' cell first.")


def timed(encode) -> list[float]:
    """Return the milliseconds `encode(tokens)` takes on each query, after a warm-up."""
    for query in queries[:20]:
        encode(json.loads(query["tokens"]))
    times = []
    for query in queries:
        started = time.perf_counter()
        encode(json.loads(query["tokens"]))
        times.append((time.perf_counter() - started) * 1000)
    return times


def bmpm_encode(tokens: list[dict]) -> None:
    """Compute the Beider–Morse tokens of every query token, in both passes."""
    for token in tokens:
        for mode in ("approx", "hebrew"):
            beider_morse[f"{token['script']}_{mode}"].encode(token["norm"])


encoding_times = {name: timed(lambda tokens, e=encode: e([t["display"] for t in tokens]))
                  for name, encode in encoders.items()}
encoding_times["l1"] = [0.0] * len(queries)
encoding_times["l4"] = timed(bmpm_encode)
_socket = start_postgres()
with psycopg.connect(host=str(_socket), port=PG_PORT, user="postgres", dbname="postgres",
                     autocommit=True) as _connection:
    _connection.execute("DROP TABLE IF EXISTS archive_forms")
    _connection.execute("CREATE TABLE archive_forms (script text, norm text, dm text[])")
    for (_script, _norm), _row in zip(spellings_to_code, name_codes):
        if _row["in_archive"]:
            _connection.execute("INSERT INTO archive_forms VALUES (%s, %s, %s)",
                                (_script, _norm, _row["dm_codes"].split()))
    _connection.execute("CREATE INDEX ON archive_forms USING gin (dm)")
    encoding_times["l2"] = timed(lambda tokens: [_connection.execute(
        "SELECT norm FROM archive_forms WHERE script = %s AND similarity(norm, %s) > 0",
        (t["script"], t["norm"])).fetchall() for t in tokens])
    encoding_times["l3"] = timed(lambda tokens: [_connection.execute(
        "SELECT norm FROM archive_forms WHERE dm && daitch_mokotoff(%s)",
        (dm_input(t["script"], t["norm"]),)).fetchall() for t in tokens])
stop_postgres()
with (OUTPUTS_DIR / "step_4_encoding_times.csv").open("w", encoding="utf-8", newline="") as _file:
    _writer = csv.writer(_file, lineterminator="\n")
    _writer.writerow(["query_id", "method", "ms"])
    for _method, _times in sorted(encoding_times.items()):
        _writer.writerows([q["query_id"], _method, f"{ms:.3f}"] for q, ms in zip(queries, _times))

print(f"{RULE}\nEncoding times: {len(queries)} queries\n{RULE}")
for _method, _times in sorted(encoding_times.items()):
    print(f"  {_method:<4} median {np.median(_times):7.2f} ms")
print(f"{RULE}\n  written to step_4_encoding_times.csv")

────────────────────────────────────────────────────────────────────────
Encoding times: 1004 queries
────────────────────────────────────────────────────────────────────────
  e1   median   56.67 ms
  e2   median    4.97 ms
  e3a  median   40.29 ms
  e3b  median   25.26 ms
  l1   median    0.00 ms
  l2   median    2.55 ms
  l3   median    0.19 ms
  l4   median    1.92 ms
────────────────────────────────────────────────────────────────────────
  written to step_4_encoding_times.csv


**5. Consistency Check**

This cell reads the outputs back and checks that every embedding file holds one unit-length vector per text, that every spelling has its phonetic codes, and that every query has a time for every method.

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Consistency check  —  the signal files against the vocabulary
# ─────────────────────────────────────────────────────────────────────────────
if "encoding_times" not in globals():
    raise RuntimeError("Run every cell above first.")

_problems = []
for _name in encoders:
    _stored = np.load(OUTPUTS_DIR / f"step_4_embeddings_{_name}.npz")
    if list(_stored["texts"]) != texts_to_embed or not np.allclose(
            np.linalg.norm(_stored["vectors"], axis=1), 1, atol=1e-3):
        _problems.append(f"step_4_embeddings_{_name}.npz differs from the vocabulary")
if len(read_csv(OUTPUTS_DIR / "step_4_name_codes.csv")) != len(spellings_to_code):
    _problems.append("step_4_name_codes.csv misses spellings")
_times = Counter(r["method"] for r in read_csv(OUTPUTS_DIR / "step_4_encoding_times.csv"))
_problems += [f"{m}: {n} times" for m, n in _times.items() if n != len(queries)]
print(f"{RULE}\nConsistency: {OUTPUTS_DIR}/step_4_*\n{RULE}")
for _problem in _problems:
    print(f"  !! {_problem}")
print(f"  {'OK' if not _problems else '!!'} {len(encoders)} embedding files, "
      f"{len(spellings_to_code)} coded spellings, {len(_times)} timed methods")
print(f"\n✓ Name signals complete ({len(texts_to_embed)} texts embedded by "
      f"{len(encoders)} models).")
print(f"  consistent: {not _problems}")

────────────────────────────────────────────────────────────────────────
Consistency: outputs/step_4_*
────────────────────────────────────────────────────────────────────────
  OK 4 embedding files, 2038 coded spellings, 8 timed methods

✓ Name signals complete (2043 texts embedded by 4 models).
  consistent: True
